# Урок 8.6. Компромиссы, развитие и карта курса

Docs Assistant работает: ищет, отвечает, цитирует, стримит, тестируется.
Финальный урок — самый взрослый навык: **честно перечислить, что мы упростили**,
понимать, когда каждое упрощение перестанет работать, и видеть в проекте
весь пройденный курс.

🎯 **В этом уроке:**

- Таблица компромиссов v1: что, почему и когда пересматривать.
- Пути масштабирования: данные, качество, нагрузка, модель.
- Карта «решение в проекте → урок курса» — финальный аккорд.
- Практика для самостоятельного развития проекта.

## 1. Компромиссы v1 — все на столе

| Упрощение | Почему сейчас ок | Сигнал пересмотреть | Куда двигаться |
|---|---|---|---|
| История в SQLite | единицы пользователей, ноль эксплуатации | конкурентная запись, несколько реплик app | PostgreSQL; интерфейс уже узкий и async (ADR-0004) |
| Нет авторизации | внутренняя сеть, один контур | выход за контур, разные роли | токены как в прокси 5.7; права на уровне поиска — модуль 4.5 |
| Переиндексация руками | документация меняется редко | жалобы «бот не знает вчерашний документ» | CI-job на мерж в репо доков; инкрементальный upsert по diff |
| Нет eval-пайплайна | история копится с первого дня | любой спор «стал хуже отвечать» | golden-набор из истории + судья из урока 3.3 |
| Нет кэша ответов | трафик мал, вопросы разные | одинаковые вопросы съедают LLM-бюджет | кэш по нормализованному вопросу с TTL |
| Один воркер uvicorn | модели в памяти одного процесса | CPU-очередь на эмбеддингах | реплики app за балансировщиком; эмбеддер — отдельный сервис |
| Нет очереди к LLM | одна Ollama, редкие запросы | 503 от Ollama под нагрузкой | семафор + очередь + потолки из прокси 5.7 (код готов!) |
| Модели грузятся в lifespan синхронно | простой и предсказуемый старт | rolling-деплой упирается в минуту старта | прогрев до включения в балансировку — это и есть /ready |

Заметьте, что **ни одно упрощение не «дыра»**: у каждого есть записанная причина,
сигнал и путь отхода. Это и отличает осознанный v1 от прототипа, дожившего
до прода.

## 2. Масштабирование по осям

**Данные** (сейчас 61 чанк). До тысяч документов масштабируется без изменений
архитектуры: батчи эмбеддингов уже есть, Qdrant с HNSW создан для миллионов
точек (урок 2.2). Меняется процесс: инкрементальная индексация (diff вместо
--recreate), фоновый воркер вместо скрипта, снапшоты Qdrant как бэкап (2.4).

**Качество.** Первый шаг всегда один: golden-набор из реальных вопросов
(история уже копит их) + двухэтапный eval из урока 3.3 (ретривер отдельно,
генератор отдельно). Дальше по результатам: гибрид BM25+RRF, если проседают
точные строки (4.4); права доступа в payload, если документация расслоится
по командам (4.5); LoRA-стиль ответов, если появится жёсткий формат (модуль 6).

**Нагрузка.** Узкие места по порядку появления: reranker на CPU (выключить
или вынести), эмбеддер (отдельный сервис с батчингом), LLM (очередь + потолки +
вторая нода Ollama — всё это мы построили в прокси 5.7 и можем поставить перед
Ollama без изменения этого проекта).

**Модель.** qwen2.5:3b заменяется строчкой конфига на любую модель Ollama —
но по процедуре из урока 5.6: пин digest, канарейка на golden-наборе, candidate-тег.
Апгрейд модели без eval — это не апгрейд, а лотерея.

## 3. План развития

- **v1.1 (неделя):** прогрев LLM в lifespan; `scripts/export_golden.py` —
  выгрузка истории в разметку; README-инструкция для Windows-политики DLL.
- **v1.2 (месяц):** авторизация по токенам + rate-limit (перенос из 5.7);
  инкрементальная индексация; eval в CI на golden-наборе.
- **v2 (по требованию):** PostgreSQL для истории; права доступа к документам;
  веб-UI со стримингом; вторая нода Ollama с балансировкой.

## 4. Карта курса в одном проекте

Финальный аккорд: каждое решение Docs Assistant — это урок курса.

| Решение в project/ | Откуда |
|---|---|
| grounding-промпт, ссылки [N], канонический отказ | 3.1, 3.4 |
| двухэтапный поиск: эмбеддинги + reranker | 2.1–2.6, 4.4 |
| Qdrant, uuid5-идемпотентность, payload-фильтры | 2.4 |
| чанкинг с сохранением заголовков секций | 2.5, 2.6 |
| e5-префиксы query:/passage: | 2.1 |
| httpx-клиент Ollama, стриминг NDJSON, таймауты | 1.4, 3.2, 5.3 |
| 127.0.0.1 вместо localhost | 3.2 (лесенка ~2 c) |
| SSE с источниками первым событием | 3.2 |
| /health против /ready, каскадные рестарты | 5.5 |
| Prometheus: гистограммы с LLM-бакетами, кардинальность | 4.3, 5.5 |
| сквозной request-id, логи без ПДн | 3.7, 4.3 |
| pydantic-settings, конфиг из окружения | 1.4 → 3.2 → сюда |
| строгие Pydantic-контракты API: extra=forbid/strict, computed-поля цитат и отказа, SSE-события как discriminated union | 3.2, 3.4, 3.8 → 8.5 |
| DI через app.state, тесты на фейках, НЕ-вызовы | 3.8, 5.7 |
| температура 0.2 для grounding | 1.2, 3.3 |
| холодный старт LLM и прогрев | 5.3 |
| «знания — в RAG, не в веса» | 6.1 |
| один агент, а не команда — по таблице цены | 7.1 |

Последняя строка — не шутка: мы **осознанно не сделали** ассистента
мульти-агентным. Задача — один связный шаг (поиск → ответ), проверяемого
выигрыша от ролей нет, а цена ×3 (урок 7.1). Знать, когда НЕ применять
модный инструмент, — тоже результат курса.

## 5. Финальная практика

Решений нет — теперь это действительно ваш проект.

1. **Свой домен.** Замените `data/docs` на свою документацию (рабочие
   регламенты, вики хобби-проекта, конспекты). Переиндексируйте, задайте
   10 вопросов, честно посчитайте: сколько ответов точны, сколько отказов
   правильных, сколько выдумок. Это ваш первый golden-набор.

2. **Замкните контур качества.** Напишите `scripts/eval.py`: читает golden
   (вопрос → ожидаемый источник), дёргает /api/search и /api/generate, считает
   hit@k ретривера и — судьёй из урока 3.3 — верность ответов. Прогоните
   с reranker и без: окупается ли он на ВАШИХ данных?

3. **Капстоун: до прода.** Поставьте перед сервисом прокси из урока 5.7
   (auth + потолки + очередь), соберите docker compose, прогоните нагрузочный
   смоук (10 параллельных клиентов — что сломается первым?), и добавьте
   в /ready проверку «модель прогрета». Когда всё зелёное — у вас в руках
   продакшен-система, собранная лично вами от токенизатора до балансировщика.

## Сводный квиз модуля

**1. Клиент /api/search получил score=0.83 и rerank_score=-1.4 у одного результата. Что из этого «правда»?**

<details><summary>Ответ</summary>

Обе правды, но про разное: 0.83 — косинус bi-encoder'а (плотная шкала e5, годна
для отбора), −1.4 — логит cross-encoder'а (растянутая шкала, годна для порядка
и уверенности). Сравнивать их между собой нельзя — потому в контракте это два
поля с предупреждением.
</details>

**2. Почему индексация — скрипт, а история — эндпоинт, а не наоборот?**

<details><summary>Ответ</summary>

По правам и частоте: индексация — редкая административная операция с записью
в базу знаний (опасно как HTTP-ручка), история — частое чтение для
пользователей и инструментов. Форма следует за моделью доступа.
</details>

**3. Ollama упала. Что произойдёт с /health, /ready, /api/search и /api/generate?**

<details><summary>Ответ</summary>

/health — 200 (процесс жив); /ready — 503 с `ollama: false`; /api/search —
работает (LLM не нужна); /api/generate — 502 через обработчик LLMError,
а в стриме — событие error. Деградация ступенчатая, а не «всё умерло».
</details>

**4. В тестах нет ни одного мока библиотеки qdrant-client. Как тогда тестируется поиск?**

<details><summary>Ответ</summary>

Подменяется НАШ сервис (FakeRag/FakeVectorStore) на границе app.state — тесты
проверяют контракт HTTP-слоя и оркестрацию, а не чужую библиотеку. Интеграцию
с настоящим Qdrant проверяет e2e (уроки 8.2–8.4) и сама эксплуатация.
</details>

**5. Чем «took_ms 7700 на первом /generate» отличается от «у нас медленный сервис»?**

<details><summary>Ответ</summary>

Это холодный старт LLM (урок 5.3): Ollama грузила модель с диска. Следующие
запросы — 1–3 с. Лечится прогревом в lifespan и отражается в /ready. Диагноз
по одному замеру — главный враг перфоманс-разговоров (урок 4.3: перцентили,
не среднее).
</details>

**6. Какие три вещи вы обязаны сделать ПЕРЕД заменой qwen2.5:3b на «модель поновее»?**

<details><summary>Ответ</summary>

Golden-набор (иначе сравнивать нечем), прогон обеих моделей на нём (ретривер
не трогаем — меняется только генератор), и пин digest новой модели с планом
отката (урок 5.6). Конфиг меняется за секунду — уверенность так быстро
не меняется.
</details>

## 🎓 Итоги модуля — и курса

Модуль 8 собрал из курса работающий продукт:

- **Проект**: FastAPI-сервис с RAG-поиском, генерацией с цитированием,
  SSE-стримингом, историей, health-пробами, метриками, тестами и Docker-упаковкой.
- **Процесс**: ТЗ → C4 → ADR → код → тесты → живые прогоны → записанные
  компромиссы. Каждый этап оставил артефакт, который переживёт автора.
- **Главный навык**: инженерная честность — знать, что упрощено, почему это
  сейчас правильно и когда перестанет быть таковым.

**Весь курс в одну строку на модуль:** токены и промпты (1) → смысл как вектор
(2) → знания в контексте, а не в весах (3) → безопасность и наблюдаемость (4) →
свой прод без облака (5) → поведение в весах (6) → команды и их цена (7) →
всё вместе, работает, протестировано (8).

**Дальше — модуль 9, послесловие курса:** глоссарий всех терминов и два
итоговых теста для самопроверки. А Docs Assistant — форкайте, наполняйте
своими знаниями и выводите в свой прод. Он ваш.